In [ ]:
## music generation
## generating new chourses

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
# loading csv file



df = pd.read_csv("/content/final_taylor_swift_lyrics.tsv", sep="\t")
df.head(40)

,index,album,song_title,lyric,line_number,release_date
0,0,Lover,Lover,[Verse 1],0,2019-08-16
1,0,Lover,Lover,We could leave the Christmas lights up 'til Ja...,1,2019-08-16
2,0,Lover,Lover,"And this is our place, we make the rules",2,2019-08-16
3,0,Lover,Lover,"And there's a dazzling haze, a mysterious way ...",3,2019-08-16
4,0,Lover,Lover,Have I known you 20 seconds or 20 years?,4,2019-08-16
5,0,Lover,Lover,[Chorus],5,2019-08-16
6,0,Lover,Lover,Can I go where you go?,6,2019-08-16
7,0,Lover,Lover,Can we always be this close?,7,2019-08-16
8,0,Lover,Lover,"Forever and ever, ah",8,2019-08-16
9,0,Lover,Lover,"Take me out, and take me home",9,2019-08-16


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16284 entries, 0 to 16283
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   index         16284 non-null  int64 
 1   album         16284 non-null  object
 2   song_title    16284 non-null  object
 3   lyric         16284 non-null  object
 4   line_number   16284 non-null  int64 
 5   release_date  11936 non-null  object
dtypes: int64(2), object(4)
memory usage: 763.4+ KB


In [4]:
df.shape

(16284, 6)

In [5]:
df.isnull().sum()

,0
index,0
album,0
song_title,0
lyric,0
line_number,0
release_date,4348


In [6]:
## no duplicates in lyric column

In [7]:
## extracting only  Chorus
df[df['lyric'].str.contains(
    r'\[Chorus',
    case=False,
    na=False
)]['lyric'].value_counts()

,count
lyric,
[Chorus],643
[Chorus 2],20
[Chorus 1],14
[Chorus: New Year's Day],6
[Chorus: Bad Blood],5
[Chorus 3],4
[Chorus: Taylor Swift],4
[Chorus - Variation],4
[Chorus: Taylor Swift & The Chicks],3


In [8]:
df = df[df['lyric'] != '''Okay so this is kind of this rough idea thing:\t2\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tUh-uh-uh-oh, oh-oh-oh-i\t3\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tOh-oh-oh, oh-oh-oh-i\t4\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tOh-oh-oh-oh, oh-oh-oh-i-i-i\t5\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tNa na na na waiting at the back door\t6\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tThey're around and the vultures come around\t7\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tAnd you know that they won't, they won't us\t8\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tSay what they want 'bout, say what they want 'bout us\t9\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tAnd you come and sing and listen\t10\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tI would not think about it but I'm not done\t11\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tEvery know that a thing or two about it, thing or two about it, I\t12\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tCause they got the cages, they got the boxes\t13\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tAnd guns\t14\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tThey are the hunters we are the foxes\t15\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tAnd we run\t16\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tBaby, I know some places we won't be found\t17\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tAnd they'll be chasing their tails trying to track us down\t18\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tCause I, I know places we can hide\t19\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tI know places, I know places\t20\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tUh-uh-uh-i, uh-uh-uh-i, oh-oh-i, uh-uh-uh-i, no places\t21\t2014-10-27\n139\t1989 (Deluxe)\tI Know Places (Voice Memo) \tOr whatever. I don't know, but that's basically just like a dark, really dark lyric, like, bridge thing and it's about, like, everybody's, like, trying to get into and ruin a love or whatever, and it's like you just, whatever, you know, and the chorus would just go to that major and just be like ooh chorus. So I hope you like it, it wouldn't be a piano thing. It would be, like, [*vocalizing*]. Like, very, like, beat. Now I'm just making weird sounds. Hope you like it!\t22\t2014-10-27\n140\tRed\tGirl at Home \t[Intro]\t0\t2012-10-22\n140\tRed\tGirl at Home \tDon't look at me, you got a girl at home\t1\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that, everybody knows that, ah-ah\t2\t2012-10-22\n140\tRed\tGirl at Home \tDon't look at me, you got a girl at home\t3\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that\t4\t2012-10-22\n140\tRed\tGirl at Home \t[Verse 1]\t5\t2012-10-22\n140\tRed\tGirl at Home \tI don't even know her\t6\t2012-10-22\n140\tRed\tGirl at Home \tBut I feel a responsibility to do what's upstanding and right\t7\t2012-10-22\n140\tRed\tGirl at Home \tIt's kinda like a code, yeah\t8\t2012-10-22\n140\tRed\tGirl at Home \tAnd you've been getting closer and closer\t9\t2012-10-22\n140\tRed\tGirl at Home \tAnd crossing so many lines\t10\t2012-10-22\n140\tRed\tGirl at Home \t[Pre-Chorus]\t11\t2012-10-22\n140\tRed\tGirl at Home \tAnd it would be a fine proposition\t12\t2012-10-22\n140\tRed\tGirl at Home \tIf I was a stupid girl\t13\t2012-10-22\n140\tRed\tGirl at Home \tBut honey, I am no one's exception\t14\t2012-10-22\n140\tRed\tGirl at Home \tThis, I have previously learned\t15\t2012-10-22\n140\tRed\tGirl at Home \t[Chorus]\t16\t2012-10-22\n140\tRed\tGirl at Home \tSo don't look at me, you got a girl at home\t17\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that, everybody knows that, ah-ah\t18\t2012-10-22\n140\tRed\tGirl at Home \tDon't look at me, you got a girl at home\t19\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that, everybody knows that\t20\t2012-10-22\n140\tRed\tGirl at Home \tI see you turn off your phone\t21\t2012-10-22\n140\tRed\tGirl at Home \tAnd now you've got me alone, and I say\t22\t2012-10-22\n140\tRed\tGirl at Home \tDon't look at me, you got a girl at home\t23\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that, everybody knows that\t24\t2012-10-22\n140\tRed\tGirl at Home \t[Verse 2]\t25\t2012-10-22\n140\tRed\tGirl at Home \tI just want to make sure\t26\t2012-10-22\n140\tRed\tGirl at Home \tYou understand perfectly\t27\t2012-10-22\n140\tRed\tGirl at Home \tYou're the kind of man who makes me sad\t28\t2012-10-22\n140\tRed\tGirl at Home \tWhile she waits up, you chase down the newest thing\t29\t2012-10-22\n140\tRed\tGirl at Home \tAnd take for granted what you have\t30\t2012-10-22\n140\tRed\tGirl at Home \t[Pre-Chorus]\t31\t2012-10-22\n140\tRed\tGirl at Home \tAnd it would be a fine proposition\t32\t2012-10-22\n140\tRed\tGirl at Home \tIf I was a stupid girl\t33\t2012-10-22\n140\tRed\tGirl at Home \tAnd, yeah, I might go with it\t34\t2012-10-22\n140\tRed\tGirl at Home \tIf I hadn't once been just like her\t35\t2012-10-22\n140\tRed\tGirl at Home \t[Chorus]\t36\t2012-10-22\n140\tRed\tGirl at Home \tSo don't look at me, you got a girl at home\t37\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that, everybody knows that, ah-ah\t38\t2012-10-22\n140\tRed\tGirl at Home \tDon't look at me, you got a girl at home\t39\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that, everybody knows that\t40\t2012-10-22\n140\tRed\tGirl at Home \tI see you turn off your phone\t41\t2012-10-22\n140\tRed\tGirl at Home \tAnd now you've got me alone, and I say\t42\t2012-10-22\n140\tRed\tGirl at Home \tDon't look at me, you got a girl at home\t43\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that, everybody knows that\t44\t2012-10-22\n140\tRed\tGirl at Home \tOh-oh-oh, oh-oh-oh\t45\t2012-10-22\n140\tRed\tGirl at Home \tOh-oh-oh, oh-oh-oh\t46\t2012-10-22\n140\tRed\tGirl at Home \t[Bridge]\t47\t2012-10-22\n140\tRed\tGirl at Home \tCall a cab, lose my number\t48\t2012-10-22\n140\tRed\tGirl at Home \tYou're about to lose your girl\t49\t2012-10-22\n140\tRed\tGirl at Home \tCall a cab, lose my number\t50\t2012-10-22\n140\tRed\tGirl at Home \tLet's consider this lesson learned\t51\t2012-10-22\n140\tRed\tGirl at Home \t[Chorus]\t52\t2012-10-22\n140\tRed\tGirl at Home \tDon't look at me, you got a girl at home\t53\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that, everybody knows that, ah-ah\t54\t2012-10-22\n140\tRed\tGirl at Home \tDon't look at me, you got a girl at home\t55\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that, everybody knows that\t56\t2012-10-22\n140\tRed\tGirl at Home \tWant to see you pick up your phone\t57\t2012-10-22\n140\tRed\tGirl at Home \tAnd tell her you're coming home\t58\t2012-10-22\n140\tRed\tGirl at Home \tDon't look at me, you got a girl at home\t59\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that, everybody knows that\t60\t2012-10-22\n140\tRed\tGirl at Home \t[Outro]\t61\t2012-10-22\n140\tRed\tGirl at Home \tDon't look at me, you got a girl at home\t62\t2012-10-22\n140\tRed\tGirl at Home \tAnd everybody knows that, everybody knows that\t63\t2012-10-22\n140\tRed\tGirl at Home \tIt would be a fine proposition\t64\t2012-10-22\n140\tRed\tGirl at Home \tIf I hadn't once been just like her\t65\t2012-10-22\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \t[Verse 1]\t0\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tI'm so glad you made time to see me\t1\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tHow's life, tell me, how's your family\t2\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tI haven't seen them in a while\t3\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tYou've been good, busier than ever\t4\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tWe small talk, work and the weather\t5\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tYour guard is up and I know why\t6\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \t[Pre-Chorus]\t7\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \t'Cause the last time you saw me\t8\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tIs still burned in the back of your mind\t9\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tYou gave me roses and I left them there to die\t10\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \t[Chorus]\t11\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tSo this is me swallowing my pride\t12\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tStanding in front of you, saying I'm sorry for that night\t13\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tAnd I'd go back to December all the time\t14\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tIt turns out freedom ain’t nothing but missing you\t15\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tWishing I'd realized what I had when you were mine\t16\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tI'd go back to December, turn around and make it alright\t17\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tI go back to December all the time\t18\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \t[Verse 2]\t19\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tThese days I haven't been sleeping\t20\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tStaying up playing back myself leaving\t21\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tWhen your birthday passed and I didn't call\t22\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tAnd I think about summer, all the beautiful times\t23\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tI watched you laughing from the passenger side and\t24\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tRealized I loved you in the fall\t25\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \t[Pre-Chorus]\t26\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tAnd then the cold came, the dark days when fear crept into my mind\t27\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tYou gave me all your love and all I gave you was goodbye\t28\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \t[Chorus]\t29\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tSo this is me swallowing my pride\t30\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tStanding in front of you, saying I'm sorry for that night\t31\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tAnd I'd go back to December all the time\t32\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tIt turns out freedom ain’t nothing but missing you\t33\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tWishing I'd realized what I had when you were mine\t34\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tI'd go back to December, turn around and change my own mind\t35\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tI go back to December all the time\t36\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \t[Bridge]\t37\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tI miss your tan skin, your sweet smile, so good to me, so right\t38\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tAnd how you held me in your arms that September night\t39\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tThe first time you ever saw me cry\t40\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tMaybe this is wishful thinking\t41\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tProbably mindless dreaming\t42\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tBut if we loved again I swear I'd love you right\t43\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tI'd go back in time and change it, but I can't\t44\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tSo if the chain is on your door, I understand...\t45\t2011-11-21\n142\tSpeak Now: World Tour Live (Deluxe)\tBack to December / Apologize / You’re Not Sorry (Live/2011) \tAnd then he said, It's too late to apologize"''']

In [9]:

df[df['lyric'].str.contains(
    r'\[Chorus',
    case=False,
    na=False
)]['lyric'].value_counts()

,count
lyric,
[Chorus],643
[Chorus 2],20
[Chorus 1],14
[Chorus: New Year's Day],6
[Chorus: Bad Blood],5
[Chorus: Taylor Swift],4
[Chorus - Variation],4
[Chorus 3],4
[Chorus: Taylor Swift & Ed Sheeran],3


In [10]:
df = df.reset_index(drop = True)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16283 entries, 0 to 16282
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   index         16283 non-null  int64 
 1   album         16283 non-null  object
 2   song_title    16283 non-null  object
 3   lyric         16283 non-null  object
 4   line_number   16283 non-null  int64 
 5   release_date  11935 non-null  object
dtypes: int64(2), object(4)
memory usage: 763.4+ KB


In [12]:
import re

chorus_lyrics = []
inside_chorus = False

for lyric in df['lyric']:

    lyric = str(lyric).strip()

    if re.search(r'^\[Chorus\b', lyric, re.IGNORECASE):
        inside_chorus = True

    elif re.match(r'^\[.*\]$', lyric):
        inside_chorus = False

    elif inside_chorus:
        chorus_lyrics.append(lyric)

print(chorus_lyrics[:20])



['Can I go where you go?', 'Can we always be this close?', 'Forever and ever, ah', 'Take me out, and take me home', "You're my, my, my, my lover", 'Can I go where you go?', 'Can we always be this close?', 'Forever and ever, ah', 'Take me out, and take me home (Forever and ever)', "You're my, my, my, my lover", 'Can I go where you go?', 'Can we always be this close?', 'Forever and ever, ah', 'Take me out, and take me home (Forever and ever)', "You're my, my, my, my", "Oh, you're my, my, my, my", "Darling, you're my, my, my, my lover", 'Ooh, look what you made me do', 'Look what you made me do', 'Look what you just made me do']


In [13]:

print("Total chorus lines:", len(chorus_lyrics))

Total chorus lines: 5798


In [14]:
## Tokenization
tokenizer = Tokenizer(char_level = False)
tokenizer.fit_on_texts(chorus_lyrics)
total_words = len(tokenizer.word_index)+1
total_words

3091

In [15]:
print(tokenizer.word_counts)

OrderedDict({'can': 154, 'i': 1771, 'go': 155, 'where': 49, 'you': 2016, 'we': 369, 'always': 63, 'be': 432, 'this': 370, 'close': 9, 'forever': 63, 'and': 1193, 'ever': 143, 'ah': 44, 'take': 97, 'me': 753, 'out': 210, 'home': 49, "you're": 250, 'my': 498, 'lover': 6, 'oh': 549, 'darling': 38, 'ooh': 151, 'look': 111, 'what': 261, 'made': 80, 'do': 196, 'just': 279, 'me—': 4, 'wanna': 117, 'your': 333, 'end': 43, 'game': 36, 'first': 66, 'string': 19, 'a': 759, 'team': 17, 'whoa': 29, 'now': 215, 'e': 24, "i'm": 292, 'the': 1365, 'only': 80, 'one': 155, 'of': 539, 'baby': 210, "that's": 124, 'fun': 23, 'eeh': 18, 'promise': 6, 'that': 523, "nobody's": 7, 'gonna': 91, 'love': 297, 'like': 306, 'let': 69, 'keep': 42, 'company': 2, 'yeah': 96, 'want': 152, 'ya': 1, 'so': 298, 'need': 91, 'to': 933, 'calm': 13, 'down': 139, 'being': 25, 'too': 68, 'loud': 10, 'stop': 19, 'not': 173, 'step': 8, 'on': 376, 'gown': 4, 'his': 38, 'our': 59, 'gowns': 2, 'in': 608, 'middle': 16, 'night': 67, 'd

In [16]:

# Convert text into numerical vectors

input_sequences = []
for line in chorus_lyrics:
  token_list = tokenizer.texts_to_sequences([line])[0]
  #print(token_list)
  for i in range(1, len(token_list)):
    input_sequences.append(token_list[:i+1])

In [17]:
print("First 10 input sequences :",input_sequences[1:10] )

print(len(input_sequences))

First 10 input sequences : [[53, 2, 50], [53, 2, 50, 152], [53, 2, 50, 152, 1], [53, 2, 50, 152, 1, 50], [53, 18], [53, 18, 125], [53, 18, 125, 14], [53, 18, 125, 14, 17], [53, 18, 125, 14, 17, 619]]
41948


In [18]:
## finding the largest length of input sequence

for x in  range(len(input_sequences)):
  if len(input_sequences[x]) == max([len(x) for x in input_sequences]):
    print("highest length of input  sequence index :" ,x)
    print("Length of input sequence : ",len(input_sequences[x]))
    print(input_sequences[x])


highest length of input  sequence index : 17684
Length of input sequence :  404
[177, 5, 269, 1, 152, 1, 66, 659, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 2, 57, 464, 1, 1710, 86, 660, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 2094, 2095, 16, 399, 86, 565, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 17, 142, 12, 734, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 22, 10, 13, 438, 1051, 9, 735, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 1051, 9, 2, 43, 119, 736, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 917, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 1383, 737, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 18, 53, 16, 394, 17, 4, 918, 738, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 16, 36, 439, 16, 17, 55, 193, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 493, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 10, 13, 438, 1711, 811, 304, 193, 271, 305, 234, 281, 237, 238, 282, 306, 29, 468, 33, 288, 467, 81, 7, 739, 304, 19

In [19]:
# pad sequences

max_seq_len = max([len(x) for x in input_sequences])


input_sequences = pad_sequences(input_sequences , maxlen = max_seq_len, padding = 'pre')
print(len(input_sequences))

41948


In [20]:
## creating X , y columns

X = input_sequences[:, :-1]

y = input_sequences[:, -1]

In [21]:

y = tf.keras.utils.to_categorical(y , num_classes = total_words)

In [22]:

print(X.shape)
print(y.shape)

(41948, 403)
(41948, 3091)


In [23]:
total_words

3091

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense ,Input

model_lstm = Sequential()

# Input layer
model_lstm.add(Input(shape=(total_words,)))

# Embedding layer
model_lstm.add(
    Embedding(
        input_dim=total_words,
        output_dim=100,
        input_length=total_words)
    )

# LSTM layer
model_lstm.add(
    LSTM(150, return_sequences=False)
    )

# Output layer
model_lstm.add(
    Dense(total_words, activation='softmax')
)

model_lstm.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 3091, 100)      │       309,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3091)           │       466,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 926,441 (3.53 MB)

 Trainable params: 926,441 (3.53 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
model_lstm.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [27]:
history = model_lstm.fit(
    X,
    y,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

Epoch 1/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.0426 - loss: 6.2446 - val_accuracy: 0.0434 - val_loss: 6.0963
Epoch 2/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.0791 - loss: 5.6157 - val_accuracy: 0.0958 - val_loss: 5.7506
Epoch 3/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 16s 27ms/step - accuracy: 0.1313 - loss: 5.0372 - val_accuracy: 0.1263 - val_loss: 5.4564
Epoch 4/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 16s 27ms/step - accuracy: 0.1752 - loss: 4.5540 - val_accuracy: 0.1373 - val_loss: 5.2733
Epoch 5/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.2193 - loss: 4.1474 - val_accuracy: 0.1638 - val_loss: 5.1455
Epoch 6/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.2670 - loss: 3.8043 - val_accuracy: 0.1914 - val_loss: 5.1001
Epoch 7/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.3104 - loss: 3.5015 - val_accuracy: 0.2131 - val_loss: 5.0375
Epoch 8/50
590/590 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.3505 - loss: 3.2331 - 

In [131]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import GRU
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Input

model_gru = Sequential()

# Input layer
model_gru.add(Input(shape=(total_words,)))

# Embedding layer
model_gru.add(
    Embedding(
        input_dim=total_words,
        output_dim=100
    )
)

# GRU layer
model_gru.add(
    GRU(150, return_sequences=False)
)

# Output layer
model_gru.add(
    Dense(total_words, activation='softmax')
)

model_gru.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 3091, 100)      │       309,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 150)            │       113,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3091)           │       466,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 889,241 (3.39 MB)

 Trainable params: 889,241 (3.39 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def generate_lyrics(seed_text, next_words, temperature=1.0):

    output_text = seed_text.lower()

    for _ in range(next_words):

        # Convert seed text into tokens
        token_list = tokenizer.texts_to_sequences(
            [output_text]
        )[0]

        # Keep only the last 10 words
        token_list = token_list[-total_words:]

        # Apply padding
        token_list = pad_sequences(
            [token_list],
            maxlen=total_words,
            padding='pre'
        )

        # Predict next word probabilities
        predictions = model_lstm.predict(
            token_list,
            verbose=0
        )[0]

        # Apply temperature
        predictions = np.log(
            np.maximum(predictions, 1e-10)
        ) / temperature

        predictions = np.exp(
            predictions - np.max(predictions)
        )

        predictions = predictions / np.sum(predictions)

        # Select next word
        predicted_index = np.random.choice(
            len(predictions),
            p=predictions
        )

        # Convert token back into word
        output_word = tokenizer.index_word.get(
            predicted_index,
            ""
        )

        if not output_word:
            break

        output_text += " " + output_word

    return output_text

In [30]:
lyrics = generate_lyrics(
    seed_text="I remember the",
    next_words=50,
    temperature=0.7
)

print(lyrics)

i remember the time who plays the water's rough my my my lover by my my lover tell me and my my lover my old and you never send it do you in a lot different all i’m fearless i think allowing yourself to cry on the bathroom floor is fearless letting go


In [31]:
print(generate_lyrics(
    "Im sorry dont leave ",
    50,
    temperature=0.8
))

im sorry dont leave  but we were dancing dancing with our ones it was happy into a spectator sport by tabloid takedown culture you had to be as excited about daily ups and what's the princess 44 2018 05 08 321 taylor swift by concert genius style love story you belong with me you'll


In [32]:
print(generate_lyrics(
    "let me love",
    50,
    temperature=1.0
))

let me love you want and all someone i do is run for you to give it people you’re not what you grew up knowing is much some of 1989 of some synth line in our most vulnerable citizens at risk and as the room right it what you know about you how
